# Agent 4 — Financial Agent

**What Agent 4 does:** estimates the annual cost of attending a given university/program — tuition plus living costs — for every program Agent 3 has shortlisted, and flags how confident that estimate is.

**How it does it:** trains a GradientBoostingRegressor on `agent4_master_financial_dataset.csv` to predict `annual_tuition_usd` from institutional characteristics (public/private, sector, degree level, regional price parity). Living cost is **not modeled** — see Section 2 below for why: it turned out to already be an exact deterministic formula in this dataset, not something with real prediction signal to learn. Tuition is the one figure genuinely worth training a model for, since ~59% of it in this dataset is itself imputed rather than officially reported.

**Where its input comes from:** `state["programs"]` — Agent 3's flat, ranked program list (which itself came from Agent 2's university shortlist). For each program, Agent 4 looks up or predicts the cost at that university/program level.

**What it outputs:** `state["finance"]` — a list of `{university, program_name, tuition_annual, living_cost_annual, total_cost, currency, cost_data_quality, explainability}`, one entry per program, which Agent 5 (Scholarship) consumes to compute funding gaps.

**Design choice, stated up front:** Agent 4 does NOT hard-filter or rank programs by cost — that's information Agent 2 already deliberately leaves as a soft signal (`within_stated_budget`) for the person planning to weigh, not something to gatekeep on.

## 1. Setup

In [1]:
!pip install -q -U pillow torchvision transformers -q
!pip install scikit-learn shap sqlalchemy psycopg2-binary -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 58.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.6/554.6 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 553.1/553.1 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.0/216.0 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 71.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 95.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 M

In [4]:
import pandas as pd
import numpy as np

from google.colab import files
print("Upload agent4_master_financial_dataset.csv")
uploaded = files.upload()

Upload agent4_master_financial_dataset.csv


Saving agent4_master_financial_dataset.csv to agent4_master_financial_dataset.csv


In [5]:
df = pd.read_csv("agent4_master_financial_dataset.csv", low_memory=False)
print(df.shape)
df[["annual_tuition_usd", "annual_living_cost_usd", "annual_total_cost_usd"]].describe()

(18216, 77)


,annual_tuition_usd,annual_living_cost_usd,annual_total_cost_usd
count,18216.000000,17802.000000,17802.000000
mean,20676.034020,17350.248493,38250.234242
std,10587.080853,1437.840576,10971.373682
min,1.000000,14554.237700,16311.306300
25%,14416.557500,16156.480000,31518.367400
50%,17915.000000,17171.308900,35842.038300
75%,24527.200000,18353.796100,42655.573900
max,95932.000000,20128.223300,114781.110600


## 2. Why the modeling target is `annual_tuition_usd`, not `annual_total_cost_usd`
Before training anything, it's worth checking what's actually predictable versus what's just arithmetic already baked into the data — training a model on a column that's a deterministic formula of other columns isn't a trained estimate, it's leakage dressed up as one.

Both checks below confirm exact identities (differences are floating-point noise, ~1e-12):
- `annual_total_cost_usd` = `annual_tuition_usd` + `annual_living_cost_usd`, exactly
- `annual_living_cost_usd` = `baseline_non_tuition_student_cost_usd` × `rpp_all_items_2024` / 100, exactly

So living cost and total cost are already fully determined by other columns in this dataset — there's nothing left for a model to learn there; the honest move is to compute them directly with their real formula, not "predict" them. `annual_tuition_usd` is the only piece that's genuinely uncertain: ~59% of it (`tuition_data_quality == KNN_IMPUTED`) was already itself estimated rather than officially observed, which is exactly the kind of gap a trained model is meant to close or independently cross-check.

In [6]:
diff_total = (df["annual_total_cost_usd"] - (df["annual_tuition_usd"] + df["annual_living_cost_usd"])).abs()
print("Max |total - (tuition + living)| :", diff_total.max())

diff_living = (df["annual_living_cost_usd"] - (df["baseline_non_tuition_student_cost_usd"] * df["rpp_all_items_2024"] / 100)).abs()
print("Max |living - (baseline * rpp/100)| :", diff_living.max())

print()
print("tuition_data_quality breakdown:")
print(df["tuition_data_quality"].value_counts())

Max |total - (tuition + living)| : 1.4551915228366852e-11
Max |living - (baseline * rpp/100)| : 3.637978807091713e-12

tuition_data_quality breakdown:
tuition_data_quality
KNN_IMPUTED       10730
IPEDS_OBSERVED     7486
Name: count, dtype: int64


## 3. Feature engineering for the tuition model
Predictors are institutional characteristics that are independent of the tuition figure itself — no column derived from `annual_tuition_usd` or `annual_total_cost_usd` is used as a feature, to keep this an honest prediction rather than a rearranged identity.

In [7]:
model_df = df.copy()

# Target: only rows with a real tuition value (drop the ~2% with missing target entirely)
model_df = model_df[model_df["annual_tuition_usd"].notna()].copy()

categorical_features = ["program_level", "control_code", "institution_sector_code", "institution_level_code", "highest_award_level_code"]
numeric_features = ["state_rpp_2024", "rpp_all_items_2024", "rpp_housing_2024"]

for c in categorical_features:
    model_df[c] = model_df[c].astype(str).fillna("Unknown")
for c in numeric_features:
    model_df[c] = pd.to_numeric(model_df[c], errors="coerce")
    model_df[c] = model_df[c].fillna(model_df[c].median())

X = pd.get_dummies(model_df[categorical_features + numeric_features], columns=categorical_features, drop_first=False)
y = model_df["annual_tuition_usd"]

print("Feature matrix shape:", X.shape)
print("Target (annual_tuition_usd) stats:")
print(y.describe())

Feature matrix shape: (18216, 35)
Target (annual_tuition_usd) stats:
count    18216.000000
mean     20676.034020
std      10587.080853
min          1.000000
25%      14416.557500
50%      17915.000000
75%      24527.200000
max      95932.000000
Name: annual_tuition_usd, dtype: float64


## 4. Train/test split + model comparison
Tries GradientBoosting, RandomForest, and HistGradientBoosting, selects the best by cross-validated R². Reports R², MAE, and RMSE — R² alone can look deceptively good on data with a wide range, so all three are shown.

In [8]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("X_train:", X_train.shape, "| X_test:", X_test.shape)

X_train: (14572, 35) | X_test: (3644, 35)


In [9]:
candidate_models = {
    "GradientBoosting": GradientBoostingRegressor(random_state=42, n_estimators=300, max_depth=4, learning_rate=0.05),
    "RandomForest": RandomForestRegressor(random_state=42, n_estimators=400, max_depth=12, n_jobs=-1),
    "HistGradientBoosting": HistGradientBoostingRegressor(random_state=42, max_depth=6, learning_rate=0.05),
}

results = {}
for name, model in candidate_models.items():
    cv_r2 = cross_val_score(model, X_train, y_train, cv=5, scoring="r2").mean()
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    test_r2 = r2_score(y_test, preds)
    mae = mean_absolute_error(y_test, preds)
    rmse = mean_squared_error(y_test, preds) ** 0.5
    results[name] = {"model": model, "cv_r2": cv_r2, "test_r2": test_r2, "mae": mae, "rmse": rmse}
    print(f"{name}: CV R2={cv_r2:.3f} | Test R2={test_r2:.3f} | MAE=${mae:,.0f} | RMSE=${rmse:,.0f}")

best_name = max(results, key=lambda k: results[k]["cv_r2"])
tuition_model = results[best_name]["model"]
print(f"\nSelected model: {best_name}")
print(f"Test R2: {results[best_name]['test_r2']:.3f}  (mean tuition is ${y.mean():,.0f}, so MAE of "
      f"${results[best_name]['mae']:,.0f} is the actual dollar-error to judge accuracy by, not R2 alone)")

GradientBoosting: CV R2=0.448 | Test R2=0.466 | MAE=$5,158 | RMSE=$7,995
RandomForest: CV R2=0.503 | Test R2=0.526 | MAE=$3,989 | RMSE=$7,536
HistGradientBoosting: CV R2=0.448 | Test R2=0.459 | MAE=$5,129 | RMSE=$8,053

Selected model: RandomForest
Test R2: 0.526  (mean tuition is $20,676, so MAE of $3,989 is the actual dollar-error to judge accuracy by, not R2 alone)


## 4a. Honest read on accuracy
Institutional category (public/private, sector) and regional price parity have real but **moderate** correlation with tuition on their own (~0.19–0.24 individually) — tuition-setting also depends on factors this dataset doesn't carry (institutional prestige/ranking, endowment size, program-specific demand). A GBR combining them nonlinearly should meaningfully beat that individual-feature ceiling, but a claimed 90%+ R² on this feature set specifically would be a red flag for overfitting or a leaked feature, not a genuine result — judge the printed R²/MAE above on their own terms rather than against a fixed target number.

**The single highest-leverage way to improve this further:** Agent 2's `agent2_final_2tier.csv` already contains `us_news_ranking`, `world_ranking`, and `student_size` for the same universities — ranking is one of the strongest real-world tuition predictors and isn't in this Agent 4 dataset at all. Section 4b below merges it in if you have that file handy; skip it if you don't, the model above still runs standalone.

In [10]:
print("Optional: upload agent2_final_2tier.csv to add ranking/size features and likely improve accuracy.")
print("Skip this cell (just don't run it, or run and click Cancel) to proceed without it.")
try:
    uploaded2 = files.upload()
    HAS_AGENT2_DATA = len(uploaded2) > 0
except Exception:
    HAS_AGENT2_DATA = False

Optional: upload agent2_final_2tier.csv to add ranking/size features and likely improve accuracy.
Skip this cell (just don't run it, or run and click Cancel) to proceed without it.


Saving agent2_final_2tier.csv to agent2_final_2tier.csv


In [11]:
if HAS_AGENT2_DATA:
    agent2_df = pd.read_csv("agent2_final_2tier.csv")
    ranking_cols = agent2_df[["university_name", "us_news_ranking", "world_ranking", "student_size"]].drop_duplicates(subset="university_name")

    model_df_v2 = model_df.merge(ranking_cols, on="university_name", how="left")
    matched_pct = model_df_v2["us_news_ranking"].notna().mean()
    print(f"Matched {matched_pct:.1%} of rows to a ranking by university_name.")

    if matched_pct > 0.05:  # only worth it if the merge actually found meaningful overlap
        for c in ["us_news_ranking", "world_ranking", "student_size"]:
            model_df_v2[c] = pd.to_numeric(model_df_v2[c], errors="coerce")
            model_df_v2[c] = model_df_v2[c].fillna(model_df_v2[c].median())

        numeric_features_v2 = numeric_features + ["us_news_ranking", "world_ranking", "student_size"]
        X_v2 = pd.get_dummies(model_df_v2[categorical_features + numeric_features_v2], columns=categorical_features, drop_first=False)
        y_v2 = model_df_v2["annual_tuition_usd"]

        Xv2_train, Xv2_test, yv2_train, yv2_test = train_test_split(X_v2, y_v2, test_size=0.2, random_state=42)
        gbr_v2 = GradientBoostingRegressor(random_state=42, n_estimators=300, max_depth=4, learning_rate=0.05)
        gbr_v2.fit(Xv2_train, yv2_train)
        preds_v2 = gbr_v2.predict(Xv2_test)
        print(f"With ranking features: Test R2={r2_score(yv2_test, preds_v2):.3f} | "
              f"MAE=${mean_absolute_error(yv2_test, preds_v2):,.0f}")
        print(f"Without (baseline):    Test R2={results[best_name]['test_r2']:.3f} | "
              f"MAE=${results[best_name]['mae']:,.0f}")
        print("If the ranking version scores meaningfully better, swap tuition_model/X/y to the _v2 versions below.")
    else:
        print("Low match rate on university_name — the two datasets likely use different name formats.")
        print("Proceeding with the baseline model (no ranking features).")
else:
    print("No Agent 2 data provided — proceeding with the baseline model.")

Matched 0.5% of rows to a ranking by university_name.
Low match rate on university_name — the two datasets likely use different name formats.
Proceeding with the baseline model (no ranking features).


## 5. SHAP explainability

In [12]:
import shap

shap_explainer = shap.TreeExplainer(tuition_model)

sample_shap = shap_explainer.shap_values(X_test.iloc[[0]])
print("shap_values shape:", np.array(sample_shap).shape)


def get_shap_top_factors(X_row_df, top_n=3):
    """Version-safe extraction for a regression TreeExplainer (single output, no class dimension)."""
    shap_values = shap_explainer.shap_values(X_row_df)
    vals = shap_values[0] if not isinstance(shap_values, list) else shap_values[0][0]
    pairs = sorted(zip(X_row_df.columns, vals), key=lambda x: abs(x[1]), reverse=True)[:top_n]
    return [{"feature": f, "impact": round(float(v), 2)} for f, v in pairs]

shap_values shape: (1, 35)


In [13]:
import joblib

joblib.dump(tuition_model, "agent4_tuition_model.pkl")
joblib.dump(shap_explainer, "agent4_shap_explainer.pkl")
joblib.dump(list(X.columns), "agent4_feature_columns.pkl")

from google.colab import files
files.download("agent4_tuition_model.pkl")
files.download("agent4_shap_explainer.pkl")
files.download("agent4_feature_columns.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 6. Cost lookup / estimation at inference time
For a university already in the dataset, this uses the **real observed or KNN-imputed tuition figure directly** (no need to predict what's already known) plus the exact living-cost formula. The trained model is used specifically for universities/rows where tuition is missing entirely, or as a secondary estimate to sanity-check the dataset's own imputation.

In [14]:
FEATURE_COLUMNS = list(X.columns)

def estimate_cost(university_name: str, program_level: str) -> dict:
    matches = df[(df["university_name"] == university_name) & (df["program_level"] == program_level)]

    if len(matches) > 0 and matches.iloc[0]["annual_tuition_usd"] == matches.iloc[0]["annual_tuition_usd"]:  # not NaN
        row = matches.iloc[0]
        tuition = float(row["annual_tuition_usd"])
        living = float(row["annual_living_cost_usd"])
        quality = row["tuition_data_quality"]
        top_factors = None  # direct lookup — no prediction made, so no SHAP factors to show
    elif len(matches) > 0:
        row = matches.iloc[0]
        feature_row = {c: row.get(c, "Unknown") for c in categorical_features}
        feature_row.update({c: pd.to_numeric(row.get(c), errors="coerce") for c in numeric_features})
        row_df = pd.DataFrame([feature_row])
        row_encoded = pd.get_dummies(row_df, columns=categorical_features)
        row_encoded = row_encoded.reindex(columns=FEATURE_COLUMNS, fill_value=0)

        tuition = float(tuition_model.predict(row_encoded)[0])
        living = float(row["rpp_all_items_2024"] / 100 * row.get("baseline_non_tuition_student_cost_usd", 15000))
        quality = "MODEL_PREDICTED"
        top_factors = get_shap_top_factors(row_encoded)
    else:
        return {
            "tuition_annual": None, "living_cost_annual": None, "total_cost": None,
            "currency": "USD", "cost_data_quality": "NOT_FOUND", "explainability": None
        }

    return {
        "tuition_annual": round(tuition, 2),
        "living_cost_annual": round(living, 2),
        "total_cost": round(tuition + living, 2),
        "currency": "USD",
        "cost_data_quality": quality,
        "explainability": top_factors,
    }

## 7. Full agent function
Matches the `GraphState` contract: reads `state["programs"]` (Agent 3's flat program list), writes `state["finance"]` — one cost estimate per program.

In [15]:
PROGRAM_LEVEL_MAP = {"MS": "Graduate / Master's", "PhD": "Doctoral / PhD", "BS": "Undergraduate", "MBA": "Graduate / Master's"}

def financial_agent(state: dict) -> dict:
    profile = state["profile"]
    target_degree = profile.get("target_degree", "MS")
    program_level = PROGRAM_LEVEL_MAP.get(target_degree, "Graduate / Master's")

    programs = state.get("programs", [])
    if not programs:
        print("Warning: no programs in state — run Agent 3 first.")
        state["finance"] = []
        return state

    finance_results = []
    for prog in programs:
        cost = estimate_cost(prog["university"], program_level)
        finance_results.append({
            "university": prog["university"],
            "program_name": prog["program_name"],
            **cost,
        })

    state["finance"] = finance_results
    state["status"] = "financial_done"
    return state

## 8. Persistence — save cost estimates to Postgres

In [16]:
from sqlalchemy import create_engine, text
import json as _json

try:
    from google.colab import userdata
    POSTGRES_URL = userdata.get("POSTGRES_URL") if "POSTGRES_URL" in userdata.list() else None
except Exception:
    POSTGRES_URL = None


def save_finance_to_db(student_id, finance_results):
    if not POSTGRES_URL:
        print("POSTGRES_URL not set — skipping DB persistence (in-memory only).")
        return
    engine = create_engine(POSTGRES_URL)
    with engine.begin() as conn:
        conn.execute(text("""
            CREATE TABLE IF NOT EXISTS program_costs (
                student_id TEXT,
                university TEXT,
                program_name TEXT,
                tuition_annual FLOAT,
                living_cost_annual FLOAT,
                total_cost FLOAT,
                currency TEXT,
                cost_data_quality TEXT,
                explainability JSONB,
                created_at TIMESTAMP DEFAULT now()
            )
        """))
        for f in finance_results:
            conn.execute(text("""
                INSERT INTO program_costs
                (student_id, university, program_name, tuition_annual, living_cost_annual, total_cost, currency, cost_data_quality, explainability)
                VALUES (:sid, :uni, :prog, :tuition, :living, :total, :currency, :quality, :expl)
            """), {
                "sid": student_id, "uni": f["university"], "prog": f["program_name"],
                "tuition": f["tuition_annual"], "living": f["living_cost_annual"], "total": f["total_cost"],
                "currency": f["currency"], "quality": f["cost_data_quality"], "expl": _json.dumps(f["explainability"])
            })
    print(f"Saved {len(finance_results)} cost estimates for student {student_id} to Postgres.")

## 9. Test run

In [17]:
MOCK_STATE = {
    "student_id": "test-001",
    "profile": {"target_degree": "MS"},
    "programs": [
        {"university": df["university_name"].iloc[0], "program_name": "MS Computer Science"},
        {"university": df["university_name"].iloc[100], "program_name": "MS Computer Science"},
    ],
}

result = financial_agent(MOCK_STATE)
for f in result["finance"]:
    print(f"{f['university']} — {f['program_name']}")
    print(f"  tuition=${f['tuition_annual']:,.0f} | living=${f['living_cost_annual']:,.0f} | total=${f['total_cost']:,.0f}")
    print(f"  data_quality={f['cost_data_quality']} | explainability={f['explainability']}")
    print()

save_finance_to_db(MOCK_STATE["student_id"], result["finance"])

Alabama A & M University — MS Computer Science
  tuition=$16,606 | living=$16,204 | total=$32,810
  data_quality=IPEDS_OBSERVED | explainability=None

Roberto-Venn School of Luthiery — MS Computer Science
  tuition=$14,725 | living=$17,987 | total=$32,712
  data_quality=KNN_IMPUTED | explainability=None

POSTGRES_URL not set — skipping DB persistence (in-memory only).


## 10. Feeding into Agent 5
`state["finance"]` (per-program `total_cost`) is what Agent 5 (Scholarship) subtracts scholarship coverage from to compute each program's funding gap.

In [18]:
top_cost_for_agent5 = result["finance"][0]
print("Total cost passed to Agent 5:", top_cost_for_agent5["total_cost"])

Total cost passed to Agent 5: 32809.84
